# Semantic Mode Evidence for Hybrid Smoothing

In [HybridInference](HybridInference.ipynb) the prior on a motion mode is a number typed into a
`DecisionTreeFactor`. A robot usually has more to go on than that: operator notes, map labels and log
lines that say something about the ground it is driving over. This notebook turns such text into a
unary `DecisionTreeFactor` on each step's mode and lets the hybrid factor graph weigh it against the
geometry.

The setup:

* A wheeled robot drives 40 steps with wheel odometry and a noisy GPS fix every 4 steps.
* Each odometry step has a binary mode $m_k \in \{\text{normal}, \text{slippery}\}$. On slippery
  ground the wheels spin, so the robot moves less than odometry reports.
* Each step also comes with a one-line note, such as *"Sprinklers ran on the lawn crossing at dawn."*

Between GPS fixes the geometry can tell that the robot fell short, but not on which steps. The notes can
say which. We compare four sources for the per-step factor $P(m_k)$:

| source | $P(m_k = \text{slippery})$ |
|---|---|
| geometry | a fixed prior of 0.25 for every step |
| keywords | 0.85 if the note matches a list of slippery words, else 0.2 |
| language model | the probability a language model gives when asked whether the note describes slippery ground |
| oracle | 0.9 or 0.1 from the true mode (an upper bound) |

The language-model probabilities come from [Jev](https://docs.typesafe.ai), TypeSafe's calibrated
decision model, following Frank Dellaert's use of it as a zero-shot source of factors for discrete
Bayes nets. They are recorded below, so the notebook runs without an API key; set `LIVE = True` to
query the model again.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/hybrid/SemanticModeEvidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass  # Not in Colab

In [2]:
import re

import numpy as np
import plotly.graph_objects as go

from gtsam import (
    DCSAM,
    BetweenFactorPose2,
    DecisionTreeFactor,
    DiscreteValues,
    HybridNonlinearFactor,
    HybridNonlinearFactorGraph,
    HybridValues,
    Pose2,
    PriorFactorPose2,
    Values,
    VectorValues,
    noiseModel,
)
from gtsam.symbol_shorthand import M, X

## Notes

Five kinds of note. Two of them are hard for keywords on purpose: slippery ground described without any
slippery word, and ordinary ground described with one ("slip ring", "dry ice"). The last slippery kind
carries no information at all, so no reader can find those steps from the text.

In [3]:
NOTES = {
    'slip_explicit': [
        'Floor in aisle 3 was mopped ten minutes ago and is still wet.',
        'Forklift oil leak near dock door 2, cones placed around the puddle.',
        'Light rain started, the outdoor path to building C is wet.',
        'Ice patch reported on the ramp by the loading bay.',
        'Spilled cleaning fluid across the corridor outside the lab.',
        'Mud tracked in from the construction site covers the entrance.',
    ],
    'slip_implicit': [
        'Sprinklers ran on the lawn crossing at dawn.',
        'Cleaning crew just finished the lobby, yellow signs still out.',
        'A pallet of olive jars broke open in aisle 4 an hour ago.',
        'Route goes through the walk-in freezer doorway.',
        'Path crosses the pebble garden by the cafeteria.',
        'Heavy dew on the grass shortcut this morning.',
    ],
    'slip_silent': [
        'Routine patrol, nothing to report.',
        'Passing the break room.',
        'Continuing to waypoint 7.',
        'Battery at 64 percent, on schedule.',
        'No obstacles detected ahead.',
        'Door to room 214 is closed.',
    ],
    'norm_plain': [
        'Carpeted corridor B, quiet.',
        'Concrete floor in warehouse zone 2.',
        'Hallway near reception, lights on.',
        'Passing the printer room.',
        'Rubber mat at the office entrance.',
        'Asphalt path between buildings A and B, dry and sunny.',
    ],
    'norm_trap': [
        'Maintenance replaced the lidar slip ring yesterday.',
        'Passing the dry ice storage cabinet, door closed.',
        'Water cooler in this hallway was refilled.',
        'Rain is forecast for tomorrow afternoon.',
        'Sand-colored carpet runner in the lobby.',
        'Grease trap inspection scheduled for next week.',
    ],
}


In [4]:
# P(slippery) returned by jev-1.13.0 for each note, recorded 2026-09-28.
RECORDED = {
    'Floor in aisle 3 was mopped ten minutes ago and is still wet.': 1.0,
    'Forklift oil leak near dock door 2, cones placed around the puddle.': 0.97,
    'Light rain started, the outdoor path to building C is wet.': 1.0,
    'Ice patch reported on the ramp by the loading bay.': 1.0,
    'Spilled cleaning fluid across the corridor outside the lab.': 1.0,
    'Mud tracked in from the construction site covers the entrance.': 1.0,
    'Sprinklers ran on the lawn crossing at dawn.': 0.98,
    'Cleaning crew just finished the lobby, yellow signs still out.': 0.99,
    'A pallet of olive jars broke open in aisle 4 an hour ago.': 0.94,
    'Route goes through the walk-in freezer doorway.': 0.56,
    'Path crosses the pebble garden by the cafeteria.': 0.98,
    'Heavy dew on the grass shortcut this morning.': 1.0,
    'Routine patrol, nothing to report.': 0.0,
    'Passing the break room.': 0.0,
    'Continuing to waypoint 7.': 0.0,
    'Battery at 64 percent, on schedule.': 0.0,
    'No obstacles detected ahead.': 0.0,
    'Door to room 214 is closed.': 0.0,
    'Carpeted corridor B, quiet.': 0.04,
    'Concrete floor in warehouse zone 2.': 0.0,
    'Hallway near reception, lights on.': 0.0,
    'Passing the printer room.': 0.0,
    'Rubber mat at the office entrance.': 0.13,
    'Asphalt path between buildings A and B, dry and sunny.': 0.0,
    'Maintenance replaced the lidar slip ring yesterday.': 0.0,
    'Passing the dry ice storage cabinet, door closed.': 0.01,
    'Water cooler in this hallway was refilled.': 0.09,
    'Rain is forecast for tomorrow afternoon.': 0.03,
    'Sand-colored carpet runner in the lobby.': 0.1,
    'Grease trap inspection scheduled for next week.': 0.14,
}

LIVE = False  # set to True to query Jev (needs `pip install typesafe-sdk` and TYPESAFE_API_KEY)

if LIVE:
    from typesafe_sdk import Choice, TypeSafeClient

    notes = [n for kind in NOTES.values() for n in kind]
    questions = {
        f"q{i}": Choice(
            instructions=f'Note for this path segment: "{n}" '
            "Which best describes the ground under the robot on this segment?",
            criteria={
                "normal": "Traction is normal, so wheel odometry is reliable.",
                "slippery": "The ground is slippery or loose, so the wheels may slip.",
            },
        )
        for i, n in enumerate(notes)
    }
    state = {
        "robot": "A wheeled delivery robot that estimates its motion from wheel odometry.",
        "task": "For each path segment, judge the ground traction from the note attached to it.",
    }
    with TypeSafeClient() as client:
        answers = client.system_one(model="jev-1.13.0", state=state, questions=questions)
    RECORDED = {n: answers.answers[f"q{i}"].probabilities["slippery"] for i, n in enumerate(notes)}

## Routes

Slippery zones of 2 to 4 steps start with probability 0.12 at each step, so about 30% of the steps
are slippery. On a slippery step the robot covers only 30 to 70% of the commanded 1 m.

In [5]:
N_STEPS, GPS_EVERY = 40, 4
TIGHT = noiseModel.Diagonal.Sigmas([0.05, 0.05, 0.02])  # normal wheel odometry
LOOSE = noiseModel.Diagonal.Sigmas([0.6, 0.6, 0.15])  # slipping wheels
GPS = noiseModel.Diagonal.Sigmas([0.15, 0.15, 10.0])  # position only


def make_route(seed):
    rng = np.random.default_rng(seed)
    modes, kinds = [], []
    while len(modes) < N_STEPS:
        if rng.random() < 0.12:
            kind = rng.choice(["slip_explicit", "slip_implicit", "slip_silent"], p=[0.4, 0.4, 0.2])
            for _ in range(min(int(rng.integers(2, 5)), N_STEPS - len(modes))):
                modes.append(1)
                kinds.append(str(kind))
        else:
            modes.append(0)
            kinds.append("norm_trap" if rng.random() < 0.3 else "norm_plain")
    notes = [str(rng.choice(NOTES[k])) for k in kinds]
    truth, odometry, gps = [Pose2()], [], {}
    for k in range(1, N_STEPS + 1):
        command = Pose2(1.0, 0.0, float(rng.choice([0.0, 0.15, -0.15])))
        if modes[k - 1]:
            actual = Pose2(command.x() * rng.uniform(0.3, 0.7), rng.normal(0, 0.1),
                           command.theta() + rng.normal(0, 0.05))
        else:
            actual = Pose2(command.x() + rng.normal(0, 0.03), rng.normal(0, 0.03),
                           command.theta() + rng.normal(0, 0.01))
        truth.append(truth[-1].compose(actual))
        odometry.append(command)
        if k % GPS_EVERY == 0:
            p = truth[k]
            gps[k] = Pose2(p.x() + rng.normal(0, 0.15), p.y() + rng.normal(0, 0.15), 0.0)
    return dict(modes=modes, kinds=kinds, notes=notes, truth=truth, odometry=odometry, gps=gps)


routes = [make_route(seed) for seed in range(40)]
route = routes[0]
for k in range(12):
    print(f"step {k + 1:2d}  {'slippery' if route['modes'][k] else 'normal  '}  {route['notes'][k]}")

step  1  normal    Grease trap inspection scheduled for next week.
step  2  slippery  Spilled cleaning fluid across the corridor outside the lab.
step  3  slippery  Spilled cleaning fluid across the corridor outside the lab.
step  4  normal    Concrete floor in warehouse zone 2.
step  5  normal    Rubber mat at the office entrance.
step  6  normal    Carpeted corridor B, quiet.
step  7  slippery  Battery at 64 percent, on schedule.
step  8  slippery  Continuing to waypoint 7.
step  9  slippery  Door to room 214 is closed.
step 10  slippery  Passing the break room.
step 11  slippery  Heavy dew on the grass shortcut this morning.
step 12  slippery  Sprinklers ran on the lawn crossing at dawn.


## Evidence

Language models answer confidently, often with probabilities of exactly 0 or 1. A factor value of 0 makes
a mode impossible, so the geometry could never overrule it. We therefore clip the probabilities to
$[\epsilon, 1-\epsilon]$ with a floor $\epsilon$ that is revisited at the end.

In [6]:
KEYWORDS = re.compile(
    r"\b(slip\w*|slick|wet|water|ice|icy|snow\w*|oil\w*|grease|greasy|mud\w*|mopped|mop|"
    r"spill\w*|puddle\w*|gravel|sand\w*|rain\w*|frost\w*|polish\w*|wax\w*|leak\w*|flood\w*|"
    r"damp|condensation|lubric\w*|slush)\b",
    re.IGNORECASE,
)


def evidence(route, source, floor=0.05):
    if source == "geometry":
        return [0.25] * N_STEPS
    if source == "keywords":
        return [0.85 if KEYWORDS.search(n) else 0.2 for n in route["notes"]]
    if source == "language model":
        return [min(max(RECORDED[n], floor), 1 - floor) for n in route["notes"]]
    if source == "oracle":
        return [0.9 if m else 0.1 for m in route["modes"]]
    raise ValueError(source)

## The hybrid factor graph

Each odometry step is a `HybridNonlinearFactor` with a tight and a loose component, as in
[HybridInference](HybridInference.ipynb). The evidence is a `DecisionTreeFactor` on the same mode, and
the GPS fixes are position priors. `DCSAM` alternates between the continuous and the discrete estimate.

In [7]:
def solve(route, p_slippery, iterations=5):
    graph, values, modes = HybridNonlinearFactorGraph(), Values(), DiscreteValues()
    graph.push_back(PriorFactorPose2(X(0), Pose2(), noiseModel.Isotropic.Sigma(3, 0.01)))
    values.insert(X(0), Pose2())
    dead_reckoning = Pose2()
    for k in range(1, N_STEPS + 1):
        mode = (M(k), 2)
        z = route["odometry"][k - 1]
        graph.push_back(HybridNonlinearFactor(mode, [BetweenFactorPose2(X(k - 1), X(k), z, TIGHT),
                                                     BetweenFactorPose2(X(k - 1), X(k), z, LOOSE)]))
        p = p_slippery[k - 1]
        graph.push_back(DecisionTreeFactor(mode, f"{1 - p} {p}"))
        if k in route["gps"]:
            graph.push_back(PriorFactorPose2(X(k), route["gps"][k], GPS))
        dead_reckoning = dead_reckoning.compose(z)
        values.insert(X(k), dead_reckoning)
        modes[M(k)] = int(p > 0.5)
    dcsam = DCSAM()
    dcsam.update(graph, HybridValues(VectorValues(), modes, values))
    for _ in range(iterations - 1):
        dcsam.update()
    result = dcsam.calculateEstimate()
    estimated_modes = [int(result.atDiscrete(M(k))) for k in range(1, N_STEPS + 1)]
    poses = [result.nonlinear().atPose2(X(k)) for k in range(N_STEPS + 1)]
    return estimated_modes, poses

## Results over 40 routes

In [8]:
SOURCES = ["geometry", "keywords", "language model", "oracle"]
results = {}
for source in SOURCES:
    rows = []
    for route in routes:
        modes, poses = solve(route, evidence(route, source))
        for k in range(N_STEPS):
            error = np.hypot(poses[k + 1].x() - route["truth"][k + 1].x(),
                             poses[k + 1].y() - route["truth"][k + 1].y())
            rows.append((route["kinds"][k], route["modes"][k], modes[k], error))
    results[source] = rows

print(f"{'source':15} {'mode accuracy':>14} {'position RMSE [m]':>18}")
for source, rows in results.items():
    accuracy = np.mean([truth == estimate for _, truth, estimate, _ in rows])
    rmse = np.sqrt(np.mean([error**2 for *_, error in rows]))
    print(f"{source:15} {accuracy:14.3f} {rmse:18.3f}")

print(f"\n{'mode accuracy by note':22}" + "".join(f"{s:>16}" for s in SOURCES))
for kind in NOTES:
    line = f"{kind:22}"
    for source in SOURCES:
        rows = [r for r in results[source] if r[0] == kind]
        line += f"{np.mean([truth == estimate for _, truth, estimate, _ in rows]):16.3f}"
    print(line)

source           mode accuracy  position RMSE [m]
geometry                 0.576              0.465
keywords                 0.660              0.452
language model           0.937              0.326
oracle                   1.000              0.160

mode accuracy by note         geometry        keywords  language model          oracle
slip_explicit                    0.985           1.000           1.000           1.000
slip_implicit                    1.000           0.491           1.000           1.000
slip_silent                      0.971           0.471           0.327           1.000
norm_plain                       0.384           0.910           0.975           1.000
norm_trap                        0.455           0.000           0.967           1.000


Geometry alone notices that the robot fell short between two GPS fixes but spreads the blame over the
whole interval. Keywords help on plainly worded notes and fail on the other two kinds by construction. The
language model reads the implicit notes and ignores the traps, which is where its gain in position error
comes from. Neither text source can do anything about notes that say nothing, and there the language
model's confident "normal" works against the geometry.

In [9]:
route = routes[1]
steps = np.arange(1, N_STEPS + 1)
fig = go.Figure()
for source in ["geometry", "keywords", "language model"]:
    _, poses = solve(route, evidence(route, source))
    error = [np.hypot(poses[k].x() - route["truth"][k].x(), poses[k].y() - route["truth"][k].y())
             for k in steps]
    fig.add_trace(go.Scatter(x=steps, y=error, mode="lines+markers", name=source))
for k, slippery in enumerate(route["modes"]):
    if slippery:
        fig.add_vrect(x0=k + 0.5, x1=k + 1.5, fillcolor="lightgray", opacity=0.5, line_width=0,
                      layer="below")
for k in route["gps"]:
    fig.add_vline(x=k, line_dash="dot", line_color="green", line_width=1)
fig.update_layout(xaxis_title="step (gray: slippery, dotted: GPS fix)",
                  yaxis_title="position error [m]", height=380)
fig.show()

## How hard to floor the language model

A higher floor keeps the text evidence from overruling the geometry when the note is uninformative, at a
small cost where the note is right.

In [10]:
print(f"{'floor':>6} {'mode accuracy':>14} {'silent-slip accuracy':>21} {'position RMSE [m]':>18}")
for floor in [0.05, 0.1, 0.2, 0.3, 0.4]:
    correct, silent, squared = [], [], []
    for route in routes:
        modes, poses = solve(route, evidence(route, "language model", floor))
        for k in range(N_STEPS):
            correct.append(modes[k] == route["modes"][k])
            if route["kinds"][k] == "slip_silent":
                silent.append(modes[k] == route["modes"][k])
            squared.append((poses[k + 1].x() - route["truth"][k + 1].x())**2
                           + (poses[k + 1].y() - route["truth"][k + 1].y())**2)
    print(f"{floor:6.2f} {np.mean(correct):14.3f} {np.mean(silent):21.3f} {np.sqrt(np.mean(squared)):18.3f}")

 floor  mode accuracy  silent-slip accuracy  position RMSE [m]
  0.05          0.937                 0.327              0.326


  0.10          0.936                 0.394              0.318
  0.20          0.935                 0.538              0.305


  0.30          0.917                 0.654              0.294
  0.40          0.882                 0.731              0.287


## Remarks

* The notes here are synthetic. What carries over is the pattern: a text source turned into a floored
  unary factor per discrete mode, fused with geometry by the hybrid graph.
* Ask the language model to compare described options ("normal" or "slippery" for this note). Numbers
  it would have to estimate, such as noise levels or how often slipping happens, belong in the model or
  should be learned from data.
* `HybridSmoother` with pruning gives an incremental alternative to `DCSAM` for long routes.